In [1]:
import pandas as pd
import ast
import json

In [2]:
df_main = pd.read_csv('../data/processed/oscar_tmdb_cleaned.csv')
credits_df = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

In [3]:
credits_df.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='str')

In [4]:
crew_keys = json.loads(credits_df['crew'][0])

crew_keys[0].keys()

dict_keys(['credit_id', 'department', 'gender', 'id', 'job', 'name'])

In [5]:
def get_director(crew_data):
    try:
        crew_list = ast.literal_eval(crew_data)
        for person in crew_list:
            if person['job'] == 'Director':
                return person['name']
        return "Unknown"
    except (ValueError, SyntaxError, TypeError):
        return "Unknown"

In [6]:
credits_df['director'] = credits_df['crew'].apply(get_director)

In [7]:
directors_df = credits_df[['movie_id', 'director']]

In [8]:
df_merged = pd.merge(df_main, directors_df, left_on='id', right_on='movie_id', how='inner')

In [9]:
df_merged.shape[0]

985

In [10]:
df_merged['director'].value_counts().head(5)

director
Steven Spielberg    23
Martin Scorsese     13
Clint Eastwood      11
Tim Burton          10
Peter Jackson        9
Name: count, dtype: int64

In [11]:
director_stats = df_merged.groupby('director')['is_oscar_winner'].mean().reset_index()

director_stats.rename(columns={'is_oscar_winner': 'director_oscar_rate'}, inplace=True)

In [12]:
director_stats.head(5)

,director,director_oscar_rate
0,Adam McKay,1.000000
1,Adrian Lyne,0.333333
2,Alan Parker,0.500000
3,Alejandro Amenábar,1.000000
4,Alejandro González Iñárritu,0.400000


In [13]:
df_final = pd.merge(df_merged, director_stats, on='director', how='left')

df_final.drop(columns=['director', 'movie_id'], inplace=True, errors='ignore')

In [14]:
df_final.to_csv('../data/processed/oscar_tmdb_with_directors.csv', index=False)